# 05 · Scoring und Bias-Metriken

Demonstration mit einem Mock-Scorer, der Position A bevorzugt; keine echten Modellresultate.


In [1]:
# --- Bootstrap: Projekt-Wurzel finden und importierbar machen ---
import sys, pathlib

PROJECT_ROOT = pathlib.Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])


PROJECT_ROOT: <Projektordner>/analysis
Python: 3.12.13


## Scorer

Echtes Scoring normalisiert die Logits des ersten Antwort-Tokens über A–D. Die Buchstaben müssen jeweils ein einzelnes, unterscheidbares Token sein. Der Mock liefert dieselbe `OptionScore`-Struktur.


In [2]:
import numpy as np
from src.models.token_scoring import OptionScore

class MockBiasedScorer:
    """Tut so, als bevorzuge das Modell Position A (Index 0) - reiner Positions-Bias."""
    def __init__(self, bias_logits=(2.0, 1.0, 0.5, 0.0)):
        self.bias = np.asarray(bias_logits, dtype=float)

    def score(self, prompts):
        out = []
        for _ in prompts:
            logits = self.bias
            probs = np.exp(logits - logits.max())
            probs = probs / probs.sum()
            out.append(OptionScore(logits=tuple(map(float, logits)),
                                   probabilities=tuple(map(float, probs))))
        return out

scorer = MockBiasedScorer()
demo = scorer.score(["irgendein prompt"])[0]
print("Wahrscheinlichkeiten A/B/C/D:", [round(x, 3) for x in demo.probabilities])
print("Vorhersage:", demo.predicted_label)


Wahrscheinlichkeiten A/B/C/D: [0.579, 0.213, 0.129, 0.078]
Vorhersage: A


## Baseline

Pro Frage entstehen Goldlabel, Vorhersage, Korrektheit sowie Logits und Wahrscheinlichkeiten. Das Beispiel benötigt kein Chat-Template.


In [3]:
from src.data.registry import load_samples
from src.utils.config import named_config, load_yaml
from src.prompts.mcq import MCQPrompt, PromptConfig
from src.evaluation.baseline import iter_baseline_batches

samples, _ = load_samples(named_config("datasets", "medqa"), "test", limit=20)
prompt = MCQPrompt(PromptConfig.from_mapping(load_yaml("configs/experiments/thesis.yaml")["prompt"]))

records = []
for batch in iter_baseline_batches(
    samples,
    scorer=scorer,
    prompt_builder=prompt,
    tokenizer=None,              # nicht noetig, weil use_chat_template=False
    use_chat_template=False,
    model_name="mock-biased-model",
    batch_size=4,
):
    records.extend(batch)

print(f"{len(records)} records gebaut. Beispiel-Felder:")
r = records[0]
print({k: r[k] for k in ("sample_id", "gold_label", "predicted_label", "correct")})
print("probs:", [round(r[f'prob_{l}'], 3) for l in 'ABCD'])


20 records gebaut. Beispiel-Felder:
{'sample_id': 'test-00000', 'gold_label': 'B', 'predicted_label': 'A', 'correct': False}
probs: [0.579, 0.213, 0.129, 0.078]


## Metriken

Accuracy, Recall je Goldlabel, Standardabweichung der vier Recalls (**RStd**) und Vorhersagehäufigkeit je Position. RStd = 0 bedeutet gleiche Recalls, beweist allein aber keine Bias-Freiheit.


In [4]:
from src.metrics.selection_bias import selection_bias_metrics

m = selection_bias_metrics(records)
print(f"n = {m['n']}   accuracy = {m['accuracy']:.3f}")
print("Recall pro richtiger Antwort:", {l: round(m[f'recall_{l}'], 3) for l in 'ABCD'})
print(f"RStd (Bias-Mass)          : {m['rstd']:.3f}")
print("Vorhersage-Haeufigkeit    :", {l: round(m[f'prediction_frequency_{l}'], 3) for l in 'ABCD'})


n = 20   accuracy = 0.050
Recall pro richtiger Antwort: {'A': 1.0, 'B': 0.0, 'C': 0.0, 'D': 0.0}
RStd (Bias-Mass)          : 0.433
Vorhersage-Haeufigkeit    : {'A': 1.0, 'B': 0.0, 'C': 0.0, 'D': 0.0}


Der Mock wählt immer A: Bei vorhandenen Goldlabels A–D ist Recall A = 1 und Recall B–D = 0. Für Unterschiede zwischen Permutationen liefert `permutation_accuracy_metrics` zusätzlich Accuracy, Varianz und Spannweite.
